In [ ]:
import os
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import numpy as np
import cv2
import tensorflow as tf
# from tensorflow import keras


import pandas as pd
import matplotlib.pyplot as plt
import json
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
train_dir = "I:/CIFAR100/Training_data"

In [ ]:
input_data = []     # input image
input_target = []   # input label
label_list = os.listdir(train_dir)
for label in label_list:
    label_path = os.path.join(train_dir, label) # ./data/training\0
    print(label_path)
    img_list = os.listdir(label_path)
    for filename in img_list:
        img_path = os.path.join(label_path, filename)
        img_ori = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img_norm = (img_ori / 255) * 0.99 + 0.01
        input_data.append(img_norm)
        input_target.append(int(label))      
 
input_data = np.array(input_data)
input_target = np.array(input_target)
print("input_data", input_data.shape)   # (2500, 32, 32, 3)
print("input_target", input_target.shape)   # (2500,)

In [ ]:

# from tensorflow import keras
# from tensorflow.keras.utils import to_categorical
input_target = tf.keras.utils.to_categorical(input_target)

print("input_data", input_data.shape)   # (2500, 32, 32, 3)
print("input_target", input_target.shape)   # (2500, 5)
print(input_target)

------------------------------------------------    切割    ------------------------------------------------

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_valid, y_train, y_valid = train_test_split(input_data, input_target, test_size=0.2, shuffle=True)
print("Training data:", x_train.shape, y_train.shape)
print("Validation data:", x_valid.shape, y_valid.shape)


In [ ]:
import visualkeras
from PIL import ImageFont
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, BatchNormalization, Dropout, Activation, AvgPool2D
# tf.keras.layers.
# from tensorflow.keras.layers import *
# input_shape=x_train.shape[1:]
#, kernel_regularizer=tf.keras.regularizers.l2(0.01)

num_classes = input_target.shape[1]

model = tf.keras.models.Sequential([
    Conv2D(filters=16, kernel_size=(3,3), padding="same", input_shape=(32,32,3)),
    BatchNormalization(),
    Activation("relu"),

    Conv2D(filters=32, kernel_size=(3,3), padding="same"),
    BatchNormalization(),
    Activation("relu"),
    MaxPool2D(),    # MaxPooling

    Dropout(0.25),  # Dropout 25%

    Conv2D(filters=64, kernel_size=(3,3), padding="same"),
    BatchNormalization(),
    Activation("relu"),

    Conv2D(filters=64, kernel_size=(3,3), padding="same"), # 不填充
    BatchNormalization(),
    Activation("relu"),
    MaxPool2D(),    # MaxPooling
    
    Dropout(0.25),  # Dropout 25%

    Conv2D(filters=128, kernel_size=(3,3), padding="same"),
    BatchNormalization(),
    Activation("relu"),

    Conv2D(filters=128, kernel_size=(3,3), padding="same"),
    BatchNormalization(),
    Activation("relu"),
    MaxPool2D(),    # MaxPooling

    Dropout(0.25),  # Dropout 25%

    Flatten(),
    Dense(256), 
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.5),  # Dropout 50%
    
    Dense(num_classes, activation="relu"),
    BatchNormalization(),
    Activation("softmax")
])
from collections import defaultdict
color_map = defaultdict(dict)
color_map[Conv2D]['fill'] = (255, 50, 50, 255)
color_map[BatchNormalization]['fill'] = 'yellow'
color_map[Activation]['fill'] = (40, 255, 40, 255)
color_map[MaxPool2D]['fill'] = 'cyan'
# color_map[Dropout]['fill'] = 'red'
color_map[Dense]['fill'] = (255, 170, 170, 255)
# visualkeras.layered_view(model, legend=True, font=ImageFont.truetype("arial.ttf", 30), to_file='model_visual.png'
#                          , color_map=color_map).show()
# tf.keras.utils.plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True)
_epoch = 200
_batch_size = 32

# # Data Augmentation
# datagen = tf.keras.preprocessing.image.ImageDataGenerator(
#     zoom_range=0.1,
#     rotation_range=10,
#     channel_shift_range=0.05,
#     width_shift_range=0.1,
#     height_shift_range=0.1,
#     horizontal_flip=True,
#     # brightness_range=(0.9, 1.1),
# )
# datagen.fit(x_train)    # not sure
# steps_per_epoch = x_train.shape[0] // _batch_size
_epoch = 200
_batch_size = 32
# iteration / epochs 
# = num of training sets / batch size 
# = 10000 * 0.8 / 32 = 250
adam = tf.keras.optimizers.Adam(
    # lr=0.001,
    lr=0.005,
    decay=1e-4,     # lr = initial_lr / (1 + 0.0001 * iterations * now_epoch)
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-08,  # avoid divided by 0
    name='Adam'
)
# opt_rms = tf.keras.optimizers.RMSprop(lr=0.001,decay=1e-6)

def get_lr_metric(optimizer):
    def lr(y_true, y_pred):
        return optimizer._decayed_lr(tf.float64) # I use ._decayed_lr method instead of .lr
    return lr
lr_metric = get_lr_metric(adam)
METRICS = [
    'accuracy',
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    lr_metric,
]

model.compile(
    loss='categorical_crossentropy',
    optimizer=adam,
    # optimizer='Adam',
    metrics=METRICS
)
# early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
# print(my_optimizer.state_dict()['param_groups'][0]['lr'])
# # 定义回调函数来打印学习率
# class PrintLearningRateCallback(tf.keras.callbacks.Callback):
#     def on_epoch_begin(self, epoch, logs=None):
#         current_learning_rate = tf.keras.backend.get_value(self.model.optimizer.lr)
#         print(f"Epoch {epoch + 1}, Learning Rate: {current_learning_rate}")
# # 定义回调函数来打印学习率
# class PrintLearningRateCallback(tf.keras.callbacks.Callback):
#     def on_epoch_end(self, epoch, logs=None):
#         current_learning_rate = self.model.optimizer.lr(self.model.optimizer.iterations).numpy()
#         print(f"\nEpoch {epoch + 1}, Learning Rate: {current_learning_rate}")
# # 创建回调函数实例
# print_lr_callback = PrintLearningRateCallback()

train_history = model.fit(
    x_train,
    y_train,
    batch_size=_batch_size,
    # datagen.flow(x=x_train, y=y_train, batch_size=_batch_size),
    # steps_per_epoch=steps_per_epoch,
    epochs=_epoch,
    validation_data=(x_valid, y_valid),
    shuffle=True,
    # callbacks=[tf.keras.callbacks.LearningRateScheduler(lr_schedule)],
    # callbacks=[print_lr_callback],
)

In [ ]:
# ----------------------------------- plot acc and loss This Training  ----------------------------------- #
# Plot the evolution of training/validation error and accuracy
evaluationT = model.evaluate(x_train, y_train, verbose=2)
evaluationV = model.evaluate(x_valid, y_valid, verbose=2)
print(f'Train Accuracy : {evaluationT[1] * 100:.3f}%')
print(f'Val Accuracy : {evaluationV[1] * 100:.3f}%')
# ----------------------------------- 
# print(train_history)
plt.figure(figsize=(12, 16))

plt.subplot(4, 2, 1)
plt.plot(train_history.history['loss'], label='Loss')
plt.plot(train_history.history['val_loss'], label='val_Loss')
plt.ylim(0, 3)  # Set the y-axis limit
plt.title('Loss Function')
plt.legend()

plt.subplot(4, 2, 2)
plt.plot(train_history.history['accuracy'], label='accuracy')
plt.plot(train_history.history['val_accuracy'], label='val_accuracy')
plt.ylim(0, 1)  # Set the y-axis limit
plt.yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])  # Set the y-axis ticks
plt.title('Accuracy Function')
plt.legend()

plt.subplot(4, 2, 3)
plt.plot(train_history.history['precision'], label='precision')
plt.plot(train_history.history['val_precision'], label='val_precision')
plt.ylim(0, 1)  # Set the y-axis limit
plt.yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])  # Set the y-axis ticks
plt.title('Precision Function')
plt.legend()

plt.subplot(4, 2, 4)
plt.plot(train_history.history['recall'], label='recall')
plt.plot(train_history.history['val_recall'], label='val_recall')
plt.ylim(0, 1)  # Set the y-axis limit
plt.yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])  # Set the y-axis ticks
plt.title('Recall Function')
plt.legend()

plt.tight_layout()
plt.show()
# 32noDA001lr
# testNoDA001lr
# test015lr
# testnoDA015lr
# 16noDA001lr
# 16noDA001lr noBN
# 1664128noDA001lr
# 161632326464noDA001lr reluBNsoft
# 32326464128128noDA001lr reluBNsoft
# 32326464128128001lr reluBNsoft
# 32326464128128015lr reluBNsoft
# 32326464128128005lr reluBNsoft

# 16noDA001lr noBN-1
# 16noDA001lr-1
# 1632noDA001lr-1
# 163264noDA001lr-1
# Before Data Augmentation


In [ ]:
# ----------------------------------- save model  ----------------------------------- #
#saveName = "cnn_image_classifier_checkpoint"               # 改這項
model.save("./outputs/models/" + saveName + ".hdf5")
# 然後將歷史信息保存到 JSON 檔案中
with open("./outputs/models/" + saveName + ".json", 'w') as file:
    json.dump(train_history.history, file)

print("train_history type:", type(train_history))
print("train_history.history type:", type(train_history.history))
print(train_history.history.keys())
print(len(train_history.history.values()))
print(len(train_history.history['loss']))
print(len(train_history.history['accuracy']))
print(len(train_history.history['val_loss']))
print(len(train_history.history['val_accuracy']))
# print(len(train_history.history.values()[0]))   # TypeError: 'dict_values' object is not subscriptable

In [ ]:
# out_model = tf.keras.models.load_model("cifar100_best_model.h5")
# out_model.summary()

In [ ]:
# ----------------------------------- load model  ----------------------------------- #
# 讀取模型
# loadName = "Cifar-10 Images Classification using CNNs (88%)_xytrain"            # 改這項

loadName = "cnn_image_classifier_checkpoint"            # 改這項
# loaded_model = tf.keras.models.load_model(loadName + ".hdf5")   # 也用於Test
loaded_model = tf.keras.models.load_model("./outputs/models/" + loadName + ".hdf5")   # 也用於Test
# loaded_model.summary()
# 加載歷史信息
# with open(loadName + ".json", 'r') as file:
with open("./outputs/models/" + loadName + ".json", 'r') as file:
    loaded_history = json.load(file)
print("loaded_history type:", type(loaded_history))
print(loaded_history.keys())
print(len(loaded_history.values()))
print(len(loaded_history['loss']))
print(len(loaded_history['accuracy']))
print(len(loaded_history['val_loss']))
print(len(loaded_history['val_accuracy']))


In [ ]:
# ----------------------------------- plot acc and loss of loaded history  ----------------------------------- #
# Plot the evolution of training/validation error and accuracy
evaluationT = model.evaluate(x_train, y_train)
evaluationV = model.evaluate(x_valid, y_valid)
print(f'Train Accuracy : {evaluationT[1] * 100:.3f}%')
print(f'Val Accuracy : {evaluationV[1] * 100:.3f}%')
# ----------------------------------- 
# print(loaded_history)
plt.figure(figsize=(12, 16))

plt.subplot(4, 2, 1)
plt.plot(loaded_history['loss'], label='Loss')
plt.plot(loaded_history['val_loss'], label='val_Loss')
plt.ylim(0, 3)  # Set the y-axis limit
plt.title('Loss Function')
plt.legend()

plt.subplot(4, 2, 2)
plt.plot(loaded_history['accuracy'], label='accuracy')
plt.plot(loaded_history['val_accuracy'], label='val_accuracy')
plt.ylim(0, 1)  # Set the y-axis limit
plt.yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])  # Set the y-axis ticks
plt.title('Accuracy Function')
plt.legend()

plt.subplot(4, 2, 3)
plt.plot(loaded_history['precision'], label='precision')
plt.plot(loaded_history['val_precision'], label='val_precision')
plt.ylim(0, 1)  # Set the y-axis limit
plt.yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])  # Set the y-axis ticks
plt.title('Precision Function')
plt.legend()

plt.subplot(4, 2, 4)
plt.plot(loaded_history['recall'], label='recall')
plt.plot(loaded_history['val_recall'], label='val_recall')
plt.ylim(0, 1)  # Set the y-axis limit
plt.yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])  # Set the y-axis ticks
plt.title('Recall Function')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# test_dir = "I:/CIFAR100/CIFAR100_testImage/my_Testing_data"
test_dir = "I:/CIFAR100/prediction_output"

test_data = []     # test image
test_file_list = os.listdir(test_dir)
test_file_list.sort(key=lambda x:int(x[:-4]))
for filename in test_file_list:
    img_path = os.path.join(test_dir, filename)
    # print(img_path)
    img_ori = cv2.imread(img_path, cv2.IMREAD_COLOR)
    img_norm = (img_ori / 255) * 0.99 + 0.01

    test_data.append(img_norm)   
 
test_data = np.array(test_data)

print("test_data", test_data.shape)   # (1000, 32, 32, 3)
print("test_data[i]", test_data[0].shape)   # (1000, 32, 32, 3)


In [ ]:
#--------------------------------------------- test from loaded model.hdf5
# tf.keras.models.Sequential.predict
predictions = model.predict(test_data)
# predictions = loaded_model.predict(test_data)
print("predictions:", predictions.shape)
print("predictions[i]:", predictions[0].shape)
predicted_classes = np.argmax(predictions, axis=1)
print("predicted_classes:", predicted_classes)
# print("predicted_classes:", predicted_classes)

# probability = (np.sum(predicted_classes[:] == 3) / len(predicted_classes[:])) * 100
# print(f"probability: {probability:.2f}%")

# sum_all = []
# for i in range(500):
# 	sum_all.append(round(sum(predictions[i]), 2))
# print(sum_all)

In [ ]:
ans_dir = "I:/CIFAR100/My_ans"
result_file = "prediction_output.txt"

with open(os.path.join(ans_dir, result_file), 'w+') as f:
    for i, png_filename in enumerate(test_file_list):
        # print("png_filename:", png_filename)
        txt_name = png_filename.split('.')[0]
        f.write("{} {}\n".format(txt_name, predicted_classes[i]))